# 🏆 [Day 36] 실전 정보 추출(IE) 및 개체명 인식(NER) 핸즈온 워크북 (ART:READY & DART 실데이터)

> **기준 문서**: [DART·ART 학습 대조 데이터 명세서 v1.0](file:///c:/Users/Playdata/enkoa-practice-knowledge-graph/enkoa-practice-knowledge-graph/내학습폴더/docs/DART_ART_학습대조_데이터명세서_v1.0.md)
>
> **핵심 학습 목표**:
> 미대 입시 모집요강(`cau_spatial_design.json`)과 기업공시에서 지식그래프 노드로 쓰일 **핵심 개체(`University`, `Department`, `AdmissionTrack`, `PracticalType`, `Company`, `Shareholder`)**를 정확히 포착하고, Pydantic 구조화 추출 및 3단계 정제 퍼널을 직접 구축·검증합니다.
>
> 1. 🧐 **[개체 표현]**: 미대 전형 문장에서 구간(Span)과 BIO 태깅의 메커니즘 및 경계 보존 실측
> 2. 📐 **[규칙 기반 매칭]**: 대학/학과/실기종목 표준 사전 대조 및 캠퍼스 분리
> 3. 🧠 **[LLM 구조화 추출]**: `ExtractedArtEntity` 스키마 기반 Pydantic 강제 타이핑
> 4. 🧪 **[3단계 정제 퍼널]**: 유형 검증 ➔ 모집요강 원문 실존 대조 ➔ 복합키 중복 제거
> 5. 🏢 **[DART 대조]**: 법인코드(`corp_code`) 기반 상장사/주주 식별성 검증

## 0. 환경 설정 및 입시 실데이터(`cau_spatial_design.json`) 로드

In [1]:
import os
import sys
import json
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Literal
from pydantic import BaseModel, Field

data_path = Path('../../내작업폴더/data/art_admission/raw/cau_spatial_design.json')
if not data_path.exists():
    data_path = Path('cau_spatial_design.json')

if data_path.exists():
    with open(data_path, 'r', encoding='utf-8') as f:
        cau_data = json.load(f)
else:
    cau_data = {
        "university": "중앙대학교", "campus": "서울", "department": "공간연출전공", "track_name": "실기형",
        "official_facts": {"exam_type_name": "1단계: 소묘(공간구성과 묘사) / 2단계: 질의응답", "admission_year": 2027}
    }

sample_text = (
    f"{cau_data['university']} {cau_data['campus']}캠퍼스 {cau_data['department']} "
    f"{cau_data['official_facts'].get('admission_year', 2027)}학년도 수시 {cau_data['track_name']} 전형은 "
    f"1단계에서 {cau_data['official_facts'].get('exam_type_name', '소묘')} 평가를 실시한다."
)
print(f"✅ [실데이터 로드 완료] {cau_data['university']} {cau_data['campus']}캠퍼스 {cau_data['department']}")
print(f'• 분석 대상 원문: "{sample_text}"')

✅ [실데이터 로드 완료] 중앙대학교 서울캠퍼스 공간연출전공
• 분석 대상 원문: "중앙대학교 서울캠퍼스 공간연출전공 2027학년도 수시 실기형 전형은 1단계에서 1단계: 소묘(공간구성과 묘사) / 2단계: 질의응답 평가를 실시한다."


## 1. 미대 입시 엔티티 구간(Span) 및 BIO 태깅 경계 보존

In [2]:
def spans_to_bio(tokens: List[str], token_spans: List[Tuple[int, int]], entity_spans: List[Tuple[int, int, str]]) -> List[str]:
    tags = ['O'] * len(tokens)
    for ent_start, ent_end, ent_type in entity_spans:
        first = True
        for idx, (t_start, t_end) in enumerate(token_spans):
            if t_start >= ent_start and t_end <= ent_end:
                tags[idx] = f'B-{ent_type}' if first else f'I-{ent_type}'
                first = False
    return tags

def decode_bio(tokens: List[str], tags: List[str]) -> List[Tuple[str, str]]:
    entities = []
    curr_tokens, curr_type = [], None
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if curr_tokens and curr_type:
                entities.append((' '.join(curr_tokens), curr_type))
            curr_tokens = [token]
            curr_type = tag.split('-')[1]
        elif tag.startswith('I-'):
            if curr_type == tag.split('-')[1]:
                curr_tokens.append(token)
        else:
            if curr_tokens and curr_type:
                entities.append((' '.join(curr_tokens), curr_type))
                curr_tokens, curr_type = [], None
    if curr_tokens and curr_type:
        entities.append((' '.join(curr_tokens), curr_type))
    return entities

tokens = ['중앙대학교', '서울캠퍼스', '공간연출전공', '실기전형', '소묘', '평가']
token_spans = [(0, 5), (6, 12), (13, 19), (20, 24), (25, 27), (28, 30)]
entity_spans = [(0, 5, 'University'), (13, 19, 'Department'), (25, 27, 'PracticalType')]

bio_tags = spans_to_bio(tokens, token_spans, entity_spans)
restored = decode_bio(tokens, bio_tags)

print(f'• 토큰 목록: {tokens}')
print(f'• BIO 태그:  {bio_tags}')
print(f'• 복원 결과: {restored}')
assert len(restored) == 3
print('✅ [PASS] 미대 입시 엔티티 경계 보존 인코딩/디코딩 검증 통과')

• 토큰 목록: ['중앙대학교', '서울캠퍼스', '공간연출전공', '실기전형', '소묘', '평가']
• BIO 태그:  ['B-University', 'O', 'B-Department', 'O', 'B-PracticalType', 'O']
• 복원 결과: [('중앙대학교', 'University'), ('공간연출전공', 'Department'), ('소묘', 'PracticalType')]
✅ [PASS] 미대 입시 엔티티 경계 보존 인코딩/디코딩 검증 통과


## 2. 규칙 기반 사전 매칭 (대학/학과/실기종목)

In [3]:
univ_dict = {'중앙대학교': 'CAU', '서울대학교': 'SNU'}
dept_dict = {'공간연출전공': 'DEPT_SPATIAL', '시각디자인과': 'DEPT_VD'}
practical_dict = {'소묘': 'PRACTICAL_DRAWING', '기초디자인': 'PRACTICAL_BASIC_DESIGN'}

matched = []
for u, c in univ_dict.items():
    if u in sample_text:
        matched.append({'name': u, 'type': 'University', 'code': c})
for d, c in dept_dict.items():
    if d in sample_text:
        matched.append({'name': d, 'type': 'Department', 'code': c})
for p, c in practical_dict.items():
    if p in sample_text:
        matched.append({'name': p, 'type': 'PracticalType', 'code': c})

print(f'• 규칙 매칭 결과 ({len(matched)}건):')
for m in matched:
    print(f"  - [{m['type']}] {m['name']} (코드: {m['code']})")

• 규칙 매칭 결과 (3건):
  - [University] 중앙대학교 (코드: CAU)
  - [Department] 공간연출전공 (코드: DEPT_SPATIAL)
  - [PracticalType] 소묘 (코드: PRACTICAL_DRAWING)


## 3. Pydantic 구조화 스키마 (`ExtractedArtEntity`)

In [4]:
ArtNodeType = Literal['University', 'Department', 'AdmissionTrack', 'PracticalType', 'ExamSchedule', 'Other']

class ExtractedArtEntity(BaseModel):
    name: str = Field(description='모집요강 원문에 등장한 명칭')
    type: ArtNodeType = Field(description='입시 노드 타입')
    campus: Optional[str] = Field(default=None, description='캠퍼스 구분')
    admission_year: Optional[int] = Field(default=2027, description='모집 학년도')

sample_entity = ExtractedArtEntity(
    name='중앙대학교',
    type='University',
    campus='서울',
    admission_year=2027
)
print('✅ [Pydantic 스키마 검증 통과] 입시 엔티티 모델 정상 동작')
print(f'• 인스턴스: {sample_entity.name} ({sample_entity.type}, campus: {sample_entity.campus}) -> 공간연출전공 (Department)')

✅ [Pydantic 스키마 검증 통과] 입시 엔티티 모델 정상 동작
• 인스턴스: 중앙대학교 (University, campus: 서울) -> 공간연출전공 (Department)


## 4. 3단계 무결성 정제 퍼널 (Post-Processing Funnel)

In [5]:
ALLOWED_ART_TYPES = {'University', 'Department', 'AdmissionTrack', 'PracticalType', 'ExamSchedule'}

def clean_art_entities(entities: List[Dict[str, Any]], raw_text: str):
    stage1 = [e for e in entities if e.get('type') in ALLOWED_ART_TYPES]
    stage2 = [e for e in stage1 if e.get('name') in raw_text]
    seen = set()
    stage3 = []
    for e in stage2:
        key = (e.get('name').strip(), e.get('type'), e.get('campus', ''))
        if key not in seen:
            seen.add(key)
            stage3.append(e)
    stats = {
        'raw': len(entities),
        'after_type': len(stage1),
        'after_presence': len(stage2),
        'final': len(stage3),
        'dropped_type': len(entities) - len(stage1),
        'dropped_hallucination': len(stage1) - len(stage2),
        'dropped_duplicates': len(stage2) - len(stage3),
    }
    return stage3, stats

mock_ents = [
    {'name': '중앙대학교', 'type': 'University', 'campus': '서울'},
    {'name': '공간연출전공', 'type': 'Department'},
    {'name': '실기형', 'type': 'AdmissionTrack'},
    {'name': '소묘', 'type': 'PracticalType'},
    {'name': '입시요강표지', 'type': 'Other'} # 정제 대상
]

clean_ents, stats = clean_art_entities(mock_ents, sample_text)
print('• [퍼널 감축 통계]')
print(f"  - 원시 추출: {stats['raw']}개")
print(f"  - 1단계 (유형 검증): {stats['after_type']}개 (탈락: {stats['dropped_type']}개)")
print(f"  - 2단계 (원문 실존): {stats['after_presence']}개 (탈락: {stats['dropped_hallucination']}개)")
print(f"  - 3단계 (중복 제거): {stats['final']}개 (탈락: {stats['dropped_duplicates']}개)")
print(f"✨ 최종 확정된 신뢰 입시 개체: {[e['name'] for e in clean_ents]}")

• [퍼널 감축 통계]
  - 원시 추출: 5개
  - 1단계 (유형 검증): 4개 (탈락: 1개)
  - 2단계 (원문 실존): 4개 (탈락: 0개)
  - 3단계 (중복 제거): 4개 (탈락: 0개)
✨ 최종 확정된 신뢰 입시 개체: ['중앙대학교', '공간연출전공', '실기형', '소묘']


## 5. DART-Trace 기업공시 복합키 식별성 대조

In [6]:
dart_corp_code = '00126380'
dart_holder = '국민연금공단'
composite_key = f'{dart_corp_code}_{dart_holder}'

print('=' * 80)
print('🏢 [DART-Trace 기업공시 복합키 식별성]')
print(f'• 법인코드: {dart_corp_code} (삼성전자)')
print(f'• 보고자키: {dart_holder}')
print(f'• 고유 복합 식별자: {composite_key} (동음이의 사명 혼동 원천 차단)')
print('=' * 80)
print('💡 [인사이트]: DART는 법인코드, ART는 {대학_캠퍼스_학과_전형_년도} 복합키로')
print('   각각의 지식그래프 노드 무결성을 완벽히 보장합니다.')

🏢 [DART-Trace 기업공시 복합키 식별성]
• 법인코드: 00126380 (삼성전자)
• 보고자키: 국민연금공단
• 고유 복합 식별자: 00126380_국민연금공단 (동음이의 사명 혼동 원천 차단)
💡 [인사이트]: DART는 법인코드, ART는 {대학_캠퍼스_학과_전형_년도} 복합키로
   각각의 지식그래프 노드 무결성을 완벽히 보장합니다.
